# Build participants.tsv from GUTBRAIN REDCAP   

## README

This script takes a redcap raw file from GUTBRAIN from MAIN GUTBRAIN report in redcap and combines it into a participants.tsv file as close as possible to the BIDS format. 

## Module loading

In [17]:
import pandas as pd
import os
from datetime import datetime
import numpy as np
import json




## Pathing

Change pathing below as needed

In [18]:

raw_redcap_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\Behavioral\data\GUTBRAIN-MainGutBrain_DATA_2025-06-03_0728.csv"
participants_outputh_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\Behavioral\data\participants.tsv"
participants_json_outputh_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\Behavioral\data\participants.json"
scratch_path = r"\\wsl.localhost\Ubuntu\home\gagpat01\mes_projets\maitrise\Behavioral\data\scratch"
os.makedirs(scratch_path, exist_ok=True)

## Raw redcap file loading and cleaning

In [19]:

# Load the raw data

df_raw_redcap = pd.read_csv(raw_redcap_path, delimiter='\t')

# Add id_recherchescan when its missing, or drop rows if it does not exist (participant dropout)

df_raw_redcap.loc[df_raw_redcap['record_id'] == 6, 'id_recherchescan'] = 'RCX097'
df_raw_redcap.loc[df_raw_redcap['record_id'] == 22, 'id_recherchescan'] = 'RCX136'
df_raw_redcap.loc[df_raw_redcap['record_id'] == 35, 'id_recherchescan'] = 'RND027'
df_raw_redcap = df_raw_redcap[df_raw_redcap['record_id'] != 67]
df_raw_redcap = df_raw_redcap[df_raw_redcap['record_id'] != 113]

# Add proper session tag 
# Create a 'session' column based on 'redcap_event_name'
df_raw_redcap['session'] = df_raw_redcap['redcap_event_name']

# Move 'session' column right after 'redcap_event_name'
cols = list(df_raw_redcap.columns)
redcap_idx = cols.index('redcap_event_name')

# Remove 'session' from the end and insert after 'redcap_event_name'
cols.insert(redcap_idx + 1, cols.pop(cols.index('session')))
df_raw_redcap = df_raw_redcap[cols]


# Use numpy.select for switch/case-like assignment

conditions = [
    (df_raw_redcap['id_recherchescan'].str[:3] == 'RGC') & (df_raw_redcap['redcap_event_name'] == '4_mois_arm_1'),
    (df_raw_redcap['id_recherchescan'].str[:3] == 'RGC') & (df_raw_redcap['redcap_event_name'] == '12_mois_arm_1'),
    (df_raw_redcap['redcap_event_name'] == 'baseline_arm_1'),
    (df_raw_redcap['redcap_event_name'] == '4_mois_arm_1'),
    (df_raw_redcap['redcap_event_name'] == '12_mois_arm_1'),
    (df_raw_redcap['redcap_event_name'] == '24_mois_arm_1'),
]
choices = [
    'x',  # RGC + 4_mois_arm_1
    '2',  # RGC + 12_mois_arm_1
    '1',  # baseline_arm_1
    '2',  # 4_mois_arm_1
    '3',  # 12_mois_arm_1
    '4',  # 24_mois_arm_1
]

df_raw_redcap['session'] = np.select(conditions, choices, default=df_raw_redcap['session'])

# Rename 'id_recherchescan' to 'participant_id'
df_raw_redcap = df_raw_redcap.rename(columns={'id_recherchescan': 'participant_id'})

# Add sub- substring to 'participant_id' to match BIDS format
df_raw_redcap['participant_id'] = 'sub-' + df_raw_redcap['participant_id'].astype(str)

# Remove record_id
df_raw_redcap = df_raw_redcap.drop(columns=['record_id'])

# Remove 'redcap_event_name' column
df_raw_redcap = df_raw_redcap.drop(columns=['redcap_event_name'])

# Reorder columns to have 'participant_id' first
cols = ['participant_id'] + [col for col in df_raw_redcap.columns if col != 'participant_id']
df_raw_redcap = df_raw_redcap[cols]

# rename 'sexe to sex'
df_raw_redcap = df_raw_redcap.rename(columns={'sexe': 'sex'})

# Normalize sex to BIDS format: "male", "female", "other"
def normalize_sex(val):
    if pd.isnull(val):
        return np.nan
    if val == 1 or str(val).strip() in ['1', 'male', 'm', 'M', 'MALE', 'Male']:
        return 'male'
    if val == 2 or str(val).strip() in ['2', 'female', 'f', 'F', 'FEMALE', 'Female']:
        return 'female'
    if str(val).strip().lower() in ['other', 'o']:
        return 'other'
    return np.nan

df_raw_redcap['sex'] = df_raw_redcap['sex'].apply(normalize_sex)

# Rename 'main_dominante' to 'handedness'
df_raw_redcap = df_raw_redcap.rename(columns={'main_dominante': 'handedness'})

# Normalize handedness to BIDS format

def normalize_handedness(val):
    if pd.isnull(val):
        return np.nan
    if val == 1 or str(val).strip() in ['1', '1.0']:
        return 'right'
    if val == 2 or str(val).strip() in ['2', '2.0']:
        return 'left'
    return np.nan

df_raw_redcap['handedness'] = df_raw_redcap['handedness'].apply(normalize_handedness)






## Debugging csv intermediates files to csv to scratch

In [20]:
# This block is for debugging, loads a preview of the data in excel scratch folder

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
preview_filename = f"df_raw_redcap_preview_{timestamp}.tsv"
preview_filepath = os.path.join(scratch_path, preview_filename)

df_raw_redcap.to_csv(preview_filepath, sep='\t', index=False)
os.system(f'start excel "{preview_filepath}"')

0

## Save file to tsv

In [21]:
df_raw_redcap.to_csv(participants_outputh_path, sep='\t', index=False)

## Create JSON file for BIDS participants



In [22]:

participants_json = {
    "participant_id": {
        "Description": "unique identifier for the participant in BIDS format",
        "Format": "sub-<id_recherchescan>"
    },
    "session": {
        "Description": "session of the participant",
        "Levels": {
            "1": "baseline",
            "2": "4 months",
            "3": "12 months",
            "4": "24 months",
            "x": "RGC second baseline"
        }
    },
    
    "age": {
        "Description": "age of the participant",
        "Units": "year"
    },
    "sex": {
        "Description": "sex of the participant",
        "Levels": {
            "M": "male",
            "F": "female"
        }
    },
    "handedness": {
        "Description": "handedness of the participant as reported by the participant",
        "Levels": {
            "left": "left",
            "right": "right"
        }
    }
    
    ,
    "t2d": {
        "Description": "Type 2 diabetes, At baseline, 1 is for diabetic, 2 is for non diabetic",
        "Levels": {
            "1": "diabetic",
            "2": "non diabetic"
        }
    },
    "type_chx": {
        "Description": "Surgery type, At baseline, 1 is for Gastrectomy, 2 is for Roux-en-Y, 3 is for DBP",
        "Levels": {
            "1": "Gastrectomy",
            "2": "Roux-en-Y",
            "3": "DBP"
        }
    },
    "dna": {
        "Description": "Birth date"
    },
    "tx_t2d___1": {
        "Description": "0 if not treated with insulin, 1 if treated with insulin",
        "Levels": {
            "0": "not treated with insulin",
            "1": "treated with insulin"
        }
    },
    "tx_t2d___2": {
        "Description": "0 if not treated with hypoglycemic treatment, 1 if treated with hypoglycemic treatment",
        "Levels": {
            "0": "not treated with hypoglycemic treatment",
            "1": "treated with hypoglycemic treatment"
        }
    },
    "tx_t2d___3": {
        "Description": "0 if not treated with diet, 1 if treated with diet",
        "Levels": {
            "0": "not treated with diet",
            "1": "treated with diet"
        }
    },
    "date_irm": {
        "Description": "Date of the scan"
    },
    "bi_ttaille_scan": {
        "Description": "Waist circumference, cm"
    },
    "bi_thanche_scan": {
        "Description": "hip circumference, cm"
    },
    "bi_tcou_scan": {
        "Description": "neck circumference, cm"
    },
    "grandeur_recherche": {
        "Description": "height for the session, cm"
    },
    "poids_bioimp_dance": {
        "Description": "weight for the session, kg"
    },
    "imc_recherche": {
        "Description": "bmi for the session"
    },
    "wl_recherche": {
        "Description": "weight loss, kg"
    },
    "ewl_recherche": {
        "Description": "effective weight loss, unitless"
    },
    "twl_recherche": {
        "Description": "total weight loss, unitless"
    }

}

json_path = participants_outputh_path.replace('participants.tsv', 'participants.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(participants_json, f, indent=4)

## Debug JSON

In [23]:
# Debugging: Save a preview of the participants.json file and open it in Chrome

timestamp_json = datetime.now().strftime("%Y%m%d_%H%M%S")
preview_json_filename = f"participants_preview_{timestamp_json}.json"
preview_json_filepath = os.path.join(scratch_path, preview_json_filename)

with open(preview_json_filepath, 'w', encoding='utf-8') as f:
    json.dump(participants_json, f, indent=4)

os.system(f'start chrome "{preview_json_filepath}"')

0

## Save JSON file to participants.json


In [24]:
with open(participants_json_outputh_path, 'w', encoding='utf-8') as f:
    json.dump(participants_json, f, indent=4)